## SmartDine — Data Ingestion Notebook

This notebook generates processed Parquet files for **train**, **val**, **test**, and an **image manifest** from your raw JSON/JSONL source that contains keys `{ "train": [...], "val": [...], "test": [...] }`.

**What it does:**
1. Load raw file from `recommender/data/raw/`  
2. Normalize to the interaction schema (`business_id`, `user_id`, `rating`, `review_text`, `pics`, `history_reviews`)  
3. Validate required columns and simple rules  
4. Write Parquet files to `recommender/data/processed/`  
5. Build `image_manifest.parquet` with unique `pic_id` values  
6. Create `MANIFEST.json` with counts, timestamp, and seed  


## 0) Configuration

In [7]:
from pathlib import Path
import os

def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Walk upward until we find a .git folder (repo root)."""
    cur = start.resolve()
    for _ in range(10):  # go up max 10 levels
        if (cur / ".git").exists():
            return cur
        cur = cur.parent
    # fallback: if not found, just use the cwd
    return start.resolve()

REPO_ROOT = find_repo_root()
print("Repo root:", REPO_ROOT)

# >>> UPDATE filename if needed
RAW_FILE = REPO_ROOT / "recommender" / "data" / "raw" / "filter_all_t.json"  # or .jsonl
OUT_DIR  = REPO_ROOT / "recommender" / "data" / "processed"

# Repro
RANDOM_SEED = int(os.environ.get("RANDOM_SEED", 42))

REQUIRED = ["business_id","user_id","rating","review_text","pics","history_reviews"]

OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_FILE, OUT_DIR



Repo root: /Users/fibonacci/Documents/fall 2025/DATA642/smartdine


(PosixPath('/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/raw/filter_all_t.json'),
 PosixPath('/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed'))

## 1) Imports

In [12]:
import json
import pandas as pd
from datetime import datetime

pd.set_option('display.max_colwidth', 120)


## 2) Helper functions


In [13]:
def load_splits(path: Path):
    """Load a JSON/JSONL file that contains one object with keys: train, val, test."""
    try:
        df = pd.read_json(path, lines=True)
        splits = df.iloc[0].to_dict()  # first row has the dict with keys
    except Exception:
        with open(path, "r") as f:
            splits = json.load(f)
    # Basic assertions
    for key in ("train","val","test"):
        if key not in splits:
            raise KeyError(f"Missing split key: {key}")
    return splits["train"], splits["val"], splits["test"]


def normalize_and_validate(records, split_name: str, required_cols=None):
    """Flatten list of dicts and ensure required columns exist."""
    required_cols = required_cols or []
    df = pd.json_normalize(records)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"{split_name} missing columns: {missing}")
    # Keep only required columns (order fixed)
    return df[required_cols]


def build_image_manifest(*dfs):
    """Collect unique pic IDs from all splits into a single manifest."""
    all_pics = []
    for d in dfs:
        # 'pics' must be a list of strings (can be empty)
        series = d["pics"].dropna()
        for item in series:
            if isinstance(item, list):
                all_pics.extend(item)
    if not all_pics:
        return pd.DataFrame(columns=["pic_id","url","local_path"])
    man = pd.DataFrame({"pic_id": pd.unique(pd.Series(all_pics))})
    man["url"] = pd.NA
    man["local_path"] = pd.NA
    return man


def write_manifest_json(out_dir: Path, counts: dict, seed: int, source_file: Path):
    meta = {
        "dataset_version": "v1",
        "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "random_seed": int(seed),
        "source_files": [str(source_file)],
        "row_counts": counts,
        "image_manifest_count": int(counts.get("image_manifest", 0)),
        "notes": ""
    }
    with open(out_dir / "MANIFEST.json", "w") as f:
        json.dump(meta, f, indent=2)
    return meta


## 3) Load raw data (train/val/test)


In [14]:
train_raw, val_raw, test_raw = load_splits(RAW_FILE)
len(train_raw), len(val_raw), len(test_raw)


(87013, 10860, 11015)

## 4) Normalize and validate to the interaction schema


In [15]:
train_df = normalize_and_validate(train_raw, "train", REQUIRED)
val_df   = normalize_and_validate(val_raw, "val", REQUIRED)
test_df  = normalize_and_validate(test_raw, "test", REQUIRED)

display(train_df.head(2))
display(val_df.head(2))
display(test_df.head(2))


,business_id,user_id,rating,review_text,pics,history_reviews
0,60567465d335d0abfb415b26,101074926318992653684,4,"The tang of the tomato sauce is outstanding. And the crust is a meal, as it should be. Order a whole pie fresh.","[AF1QipM-2IRmvitARbcJr7deWfe5hyVBg_ArPMQSYvq0, AF1QipPWhe1OP80YPU40J6-XIdxbJIe57vKm8TTjve31, AF1QipNuKWM65S9ZFQykvdI...","[[101074926318992653684_6056272797d555cc6fb0d147, The pizza here is the real deal, perfect in every way except for t..."
1,6050fa9f5b4ccec8d5cae994,117065749986299237881,5,Chicken and waffles were really good!,[AF1QipMpfxIZUT_aymQ3qPGO-QgGYzxbtLZGmHufAp2s],"[[117065749986299237881_605206f8d8c08f462b93e826, This was the Chipotle Deviled Eggs.]]"


,business_id,user_id,rating,review_text,pics,history_reviews
0,6049974fb1a0aaee3eefb0dd,112777069092124620875,5,It's really the best hot chicken I've had anywhere.,[AF1QipNhbk-hwCwq2O6JBZQq6UXgIpwtzr-tQFTKxMIG],"[[112777069092124620875_6043a4a88be5d4454df9dd76, We hand the Amish Chicken and Hangar Steak. Plus to start some Laz..."
1,6040f95d7cd8bf1303622198,116435353904842843088,5,The omelettes are really good but come hungry because they are huge. 6 egg omelettes with maximum stuffing. The bana...,"[AF1QipNqBN9aY7-jiGbrBfOFGK1gZsjWSV7pgPR0cvA-, AF1QipPyyQOtRLb4rRL8D-cXCKxh4rLP3fJL8tm0NzGG]","[[116435353904842843088_60571bb4f69c7b117807031a, Seriously one of the best burgers Ive had. Simple and delicious th..."


,business_id,user_id,rating,review_text,pics,history_reviews
0,604bf6a75041fa50c4bce594,108919790647235091207,5,This was my first time having sushi and I definitely picked the right place.,"[AF1QipNb7nd3nww6uVOe9MaJzvrkiewlININEKYaXfRm, AF1QipNehwG5eEorYu1JxGlUAOLjp7P1YNKYsPJ9nDLk, AF1QipMVkejQ2gFiyUXVVT1...","[[108919790647235091207_6055eda33019cb0a47838b25, I had the veggie burger and it was okay. I also had the mango keyl..."
1,604ee8b388c7af3f893e613b,108111397722253060630,5,I tried the spicy 'Nashville Hot Chicken'.,"[AF1QipMQwmNbcM2swmTNjzJWrwpnkOrUiKPcFsFdo_EO, AF1QipNYI-T7dWvmSzl9sXOZVaY3-PB001HOC8ZT-yUH]","[[108111397722253060630_604bca76d40e4bc9b841777c, Amazing and delicious pizzas and Pretzel bites .], [10811139772225..."


## 5) Simple QA checks


In [16]:
qa = {
    "train_columns_ok": list(train_df.columns) == REQUIRED,
    "val_columns_ok": list(val_df.columns) == REQUIRED,
    "test_columns_ok": list(test_df.columns) == REQUIRED,
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "test_rows": len(test_df),
}
qa


{'train_columns_ok': True,
 'val_columns_ok': True,
 'test_columns_ok': True,
 'train_rows': 87013,
 'val_rows': 10860,
 'test_rows': 11015}

## 6) Write Parquet files


In [17]:
train_path = OUT_DIR / "train.parquet"
val_path   = OUT_DIR / "val.parquet"
test_path  = OUT_DIR / "test.parquet"

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

[str(p) for p in (train_path, val_path, test_path)]


['/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/train.parquet',
 '/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/val.parquet',
 '/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/test.parquet']

## 7) Build image_manifest.parquet


In [18]:
manifest_df = build_image_manifest(train_df, val_df, test_df)
manifest_path = OUT_DIR / "image_manifest.parquet"
manifest_df.to_parquet(manifest_path, index=False)

manifest_df.head(), len(manifest_df), str(manifest_path)



(                                         pic_id   url local_path
 0  AF1QipM-2IRmvitARbcJr7deWfe5hyVBg_ArPMQSYvq0  <NA>       <NA>
 1  AF1QipPWhe1OP80YPU40J6-XIdxbJIe57vKm8TTjve31  <NA>       <NA>
 2  AF1QipNuKWM65S9ZFQykvdIhKUliE6K1VBxssTUYyl8d  <NA>       <NA>
 3  AF1QipOJng1JS_1hmpfhAVrr7hE89dcoOtdy-Z6cOO9x  <NA>       <NA>
 4  AF1QipMpfxIZUT_aymQ3qPGO-QgGYzxbtLZGmHufAp2s  <NA>       <NA>,
 202133,
 '/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/image_manifest.parquet')

## 8) Write MANIFEST.json (provenance)


In [19]:
counts = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
    "image_manifest": len(manifest_df),
}
meta = write_manifest_json(OUT_DIR, counts, RANDOM_SEED, RAW_FILE)
meta


/var/folders/vz/g45t4ljn0qj6yqn06cb_w6bc0000gn/T/ipykernel_84610/1577454553.py:47: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",


{'dataset_version': 'v1',
 'created_at_utc': '2025-10-30T04:12:56Z',
 'random_seed': 42,
 'source_files': ['/Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/raw/filter_all_t.json'],
 'row_counts': {'train': 87013,
  'val': 10860,
  'test': 11015,
  'image_manifest': 202133},
 'image_manifest_count': 202133,
 'notes': ''}

## 9) Summary


In [20]:
print("Wrote:")
print(" -", OUT_DIR / "train.parquet")
print(" -", OUT_DIR / "val.parquet")
print(" -", OUT_DIR / "test.parquet")
print(" -", OUT_DIR / "image_manifest.parquet")
print(" -", OUT_DIR / "MANIFEST.json")


Wrote:
 - /Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/train.parquet
 - /Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/val.parquet
 - /Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/test.parquet
 - /Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/image_manifest.parquet
 - /Users/fibonacci/Documents/fall 2025/DATA642/smartdine/recommender/data/processed/MANIFEST.json
